#### **DADOS DA CVM E B3: ANUAIS E TRIMESTRAIS**

**RESUMO**  
Apresenta o roteiro direto para baixar os dados públicos da Comissão de Valores Mobiliários (CVM) referentes às companhias de capital aberto negociadas na B3. Consideramos as bases de dados em termos anuais e trimestrais.

-------------------------------------------------------------------------------
**AUTOR**: Prof. Dr. Elson Rodrigo de Souza Santos  
**E-MAIL**: [elson129@gmail.com] ou [elson.rodrigo@ufabc.edu.br]  
**REPOSITÓRIO**: https://github.com/elson29r/model_data_general  
**DATA**: Agosto de 2026


**Sumário**
1) Instalação dos pacotes
2) Código e teste: Petrobras; Vale; e Banco do Brasil
3) Base de dados completa (anual e trimestral)

**1) Instalação dos pacotes:**  

Pacotes utilizados:

- **Pandas** - análise e manutenção de em Python. O pacote fornece estruturas de dados eficientes, chamadas DataFrames, ideais para organizar painéis longitudinais, tratar séries temporais e preparar a matriz final de variáveis;

- **Requests** - biblioteca Python para realizar requisições HTTP;

- **Pyarrow** - comprimir, organizar e escrever os dados.

In [ ]:
# 1.1 Instalação

!pip install pandas requests pyarrow

**2) Código e teste: Petrobras; Vale; e Banco do Brasil**  

Apresentamos o código base para baixar os dados. Em seguida, realizamos o teste para as empresas de capital aberto selecionadas: Petrobras, Vale e Banco do Brasil. A escolha dessas empresas ocorre devido a serem tradicionais na bolsa de valores, elevadas em valor de mercado e quantidade de ações, estarem entre as mais negociadas na bolsa e de maior liquidez.


**Sumário**  
2.1 Código das firmas (Petrobras, Vale e Banco do Brasil)  
2.2 Baixa os dados anuais  
2.3 Baixa os dados trimestrais  
2.4 Exportação (anual e trimestral)

In [ ]:
# 2.1 Código das firmas (Petrobras, Vale e Banco do Brasil)  

import io
import pandas as pd
import requests

BASE_CAD_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/CAD/DADOS/cad_cia_aberta.csv"

def baixar_cadastro() -> pd.DataFrame:
    resposta = requests.get(BASE_CAD_URL, timeout=60)
    resposta.raise_for_status()
    return pd.read_csv(io.BytesIO(resposta.content), sep=";", encoding="latin1")


def buscar_codigo_cvm(cadastro: pd.DataFrame, termo: str) -> pd.DataFrame:
    filtro = cadastro["DENOM_SOCIAL"].str.contains(termo, case=False, na=False)
    return cadastro.loc[filtro, ["CD_CVM", "DENOM_SOCIAL", "SIT"]]


if __name__ == "__main__":
    cadastro = baixar_cadastro()
    for termo in ["PETROBRAS", "VALE", "BANCO DO BRASIL"]:
        print(buscar_codigo_cvm(cadastro, termo))

# Obs.: estamos procurando o código das empresas alvo. Podem ser modificados para qualquer empresa específica da base da CVM



In [ ]:
# 2.2 Baixa os dados anuais

# Obs.: utilizamos como base as firmas: Petrobras; Vale; e Banco do Brasil

import io
import zipfile
from pathlib import Path
from typing import Iterable
import pandas as pd
import requests

BASE_DFP_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS"


def baixar_zip_dfp(ano: int, cache_folder: Path) -> Path:
    cache_folder.mkdir(parents=True, exist_ok=True)
    destino = cache_folder / f"dfp_cia_aberta_{ano}.zip"
    if destino.exists():
        return destino
    url = f"{BASE_DFP_URL}/dfp_cia_aberta_{ano}.zip"
    resposta = requests.get(url, timeout=60)
    resposta.raise_for_status()
    destino.write_bytes(resposta.content)
    return destino


def ler_csv_do_zip(
    caminho_zip: Path,
    nome_arquivo: str,
    codigos_cvm: Iterable[int] | None = None,
) -> pd.DataFrame:
    with zipfile.ZipFile(caminho_zip) as arquivo_zip:
        if nome_arquivo not in arquivo_zip.namelist():
            return pd.DataFrame()
        with arquivo_zip.open(nome_arquivo) as csv_bytes:
            leitor = pd.read_csv(
                io.BytesIO(csv_bytes.read()),
                sep=";",
                encoding="latin1",
                decimal=",",
                dtype={"CD_CVM": "Int64"},
                chunksize=200_000,  # filtra por partes, não carrega o CSV inteiro
            )
            partes = [
                parte[parte["CD_CVM"].isin(codigos_cvm)] if codigos_cvm is not None else parte
                for parte in leitor
            ]
    if not partes:
        return pd.DataFrame()
    return pd.concat(partes, ignore_index=True)


def get_dfp_data(
    companies_cvm_codes: Iterable[int] | None = None,
    first_year: int = 2011,
    last_year: int = 2025,
    type_docs: list[str] = ["BPA", "BPP", "DRE"],
    type_format: list[str] = ["con", "ind"],
    cache_folder: str = "cvm_dfp_cache",
) -> dict[str, pd.DataFrame]:
    cache_path = Path(cache_folder)
    resultado: dict[str, list[pd.DataFrame]] = {}
    for ano in range(first_year, last_year + 1):
        try:
            caminho_zip = baixar_zip_dfp(ano, cache_path)
        except requests.HTTPError:
            print(f"Ano {ano} indisponível no portal da CVM, pulando.")
            continue
        for tipo in type_docs:
            for formato in type_format:
                nome_arquivo = f"dfp_cia_aberta_{tipo}_{formato}_{ano}.csv"
                chave = f"{tipo}_{formato}"
                df_ano = ler_csv_do_zip(caminho_zip, nome_arquivo, companies_cvm_codes)
                if df_ano.empty:
                    continue
                resultado.setdefault(chave, []).append(df_ano)
    return {
        chave: pd.concat(lista, ignore_index=True)
        for chave, lista in resultado.items()
    }


if __name__ == "__main__":
    codigos_cvm = [9512, 4170, 1023]  # Petrobras, Vale e BB
    dados_dfp = get_dfp_data(
        companies_cvm_codes=codigos_cvm,
        first_year=2011,
        last_year=2025,
        type_docs=["DRE", "BPA", "BPP"],
        type_format=["con"],
    )
    for chave, df in dados_dfp.items():
        print(f"{chave}: {df.shape[0]} linhas, {df.shape[1]} colunas")

In [ ]:
# 2.3 Baixa os dados trimestrais

# Obs.: utilizamos como base as firmas: Petrobras; Vale; e Banco do Brasil

import io
import zipfile
from pathlib import Path
from typing import Iterable
import pandas as pd
import requests

BASE_ITR_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/ITR/DADOS"


def baixar_zip_itr(ano: int, cache_folder: Path) -> Path:
    cache_folder.mkdir(parents=True, exist_ok=True)
    destino = cache_folder / f"itr_cia_aberta_{ano}.zip"
    if destino.exists():
        return destino
    url = f"{BASE_ITR_URL}/itr_cia_aberta_{ano}.zip"
    resposta = requests.get(url, timeout=60)
    resposta.raise_for_status()
    destino.write_bytes(resposta.content)
    return destino


def ler_csv_do_zip(caminho_zip: Path, nome_arquivo: str) -> pd.DataFrame:
    with zipfile.ZipFile(caminho_zip) as arquivo_zip:
        if nome_arquivo not in arquivo_zip.namelist():
            return pd.DataFrame()
        with arquivo_zip.open(nome_arquivo) as csv_bytes:
            df = pd.read_csv(
                io.BytesIO(csv_bytes.read()),
                sep=";",
                encoding="latin1",
                decimal=",",
                dtype={"CD_CVM": "Int64"},
            )
    return df


def get_itr_data(
    companies_cvm_codes: Iterable[int] | None = None,
    first_year: int = 2011,
    last_year: int = 2025,
    type_docs: list[str] = ["BPA", "BPP", "DRE"],
    type_format: list[str] = ["con", "ind"],
    cache_folder: str = "cvm_itr_cache",
) -> dict[str, pd.DataFrame]:
    cache_path = Path(cache_folder)
    resultado: dict[str, list[pd.DataFrame]] = {}
    for ano in range(first_year, last_year + 1):
        try:
            caminho_zip = baixar_zip_itr(ano, cache_path)
        except requests.HTTPError:
            print(f"Ano {ano} indisponível no portal da CVM, pulando.")
            continue
        for tipo in type_docs:
            for formato in type_format:
                nome_arquivo = f"itr_cia_aberta_{tipo}_{formato}_{ano}.csv"
                chave = f"{tipo}_{formato}"
                df_ano = ler_csv_do_zip(caminho_zip, nome_arquivo)
                if df_ano.empty:
                    continue
                if companies_cvm_codes is not None:
                    df_ano = df_ano[df_ano["CD_CVM"].isin(companies_cvm_codes)]
                resultado.setdefault(chave, []).append(df_ano)
    return {
        chave: pd.concat(lista, ignore_index=True)
        for chave, lista in resultado.items()
    }


if __name__ == "__main__":
    codigos_cvm = [9512, 4170, 1023]  # Petrobras, Vale e BB

    dados_itr = get_itr_data(
        companies_cvm_codes=codigos_cvm,
        first_year=2011,
        last_year=2025,
        type_docs=["DRE", "BPA", "BPP"],
        type_format=["con"],
    )

    for chave, df in dados_itr.items():
        print(f"{chave}: {df.shape[0]} linhas, {df.shape[1]} colunas")

In [ ]:
# 2.4 Exportação (anual e trimestral)

from pathlib import Path

import pandas as pd


def exportar_para_csv(dados: dict[str, pd.DataFrame], pasta: str) -> None:
    if not dados:
        print(f"Nenhum dado para exportar em {pasta}.")
        return

    Path(pasta).mkdir(parents=True, exist_ok=True)
    for chave, df in dados.items():
        caminho = Path(pasta) / f"{chave}.csv"
        df.to_csv(caminho, index=False)
        print(f"{chave}: {df.shape[0]} linhas exportadas para {caminho}")


if __name__ == "__main__":
    exportar_para_csv(dados_itr, "dados_trimestrais_firmasalvo")
    exportar_para_csv(dados_dfp, "dados_anuais_firmasalvo")

# Obs.: cada chave do dicionário (ex.: DRE_con, BPA_con) vira um
# arquivo CSV separado, dentro das pastas indicadas

#### **3) Base de dados completa (anual e trimestral)**

Extraímos a base de dados completa da CVM das empresas ativas na B3

**Sumário**  
3.1 Dados anuais completo  
3.2 Dados trimestrais completo  
3.3 Exportação (anual e trimestral)

In [ ]:
# 3.1 Dados anuais completo

import zipfile
from pathlib import Path
from typing import Iterable

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
import io
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_DFP_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS"
BASE_CAD_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/CAD/DADOS/cad_cia_aberta.csv"


def criar_sessao() -> requests.Session:
    """Sessão HTTP com novas tentativas automáticas em caso de falha
    de rede, o que reduz travamentos por instabilidade de conexão."""
    sessao = requests.Session()
    retentativas = Retry(
        total=5,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
    )
    sessao.mount("https://", HTTPAdapter(max_retries=retentativas))
    return sessao


def obter_codigos_ativos(sessao: requests.Session) -> list[int]:
    resposta = sessao.get(BASE_CAD_URL, timeout=60)
    resposta.raise_for_status()
    cadastro = pd.read_csv(
        io.StringIO(resposta.content.decode("latin1")),
        sep=";",
    )
    ativas = cadastro.loc[cadastro["SIT"] == "ATIVO", "CD_CVM"]
    return ativas.unique().tolist()


def baixar_zip_dfp(ano: int, cache_folder: Path, sessao: requests.Session) -> Path:
    """Baixa o zip em streaming, gravando por blocos para não carregar
    o arquivo inteiro na memória de uma só vez."""
    cache_folder.mkdir(parents=True, exist_ok=True)
    destino = cache_folder / f"dfp_cia_aberta_{ano}.zip"
    if destino.exists():
        return destino

    url = f"{BASE_DFP_URL}/dfp_cia_aberta_{ano}.zip"
    with sessao.get(url, timeout=60, stream=True) as resposta:
        resposta.raise_for_status()
        with open(destino, "wb") as arquivo:
            for bloco in resposta.iter_content(chunk_size=1024 * 1024):
                arquivo.write(bloco)

    return destino


def ler_csv_do_zip(
    caminho_zip: Path,
    nome_arquivo: str,
    codigos_cvm: set[int],
) -> pd.DataFrame:
    with zipfile.ZipFile(caminho_zip) as arquivo_zip:
        if nome_arquivo not in arquivo_zip.namelist():
            return pd.DataFrame()

        with arquivo_zip.open(nome_arquivo) as csv_bytes:
            leitor = pd.read_csv(
                csv_bytes,
                sep=";",
                encoding="latin1",
                decimal=",",
                dtype={"CD_CVM": "Int64"},
                chunksize=200_000,
            )
            partes = [
                parte[parte["CD_CVM"].isin(codigos_cvm)] for parte in leitor
            ]

    if not partes:
        return pd.DataFrame()

    return pd.concat(partes, ignore_index=True)


def get_dfp_data(
    companies_cvm_codes: Iterable[int],
    first_year: int = 2011,
    last_year: int = 2025,
    type_docs: list[str] = ["BPA", "BPP", "DRE"],
    type_format: list[str] = ["con", "ind"],
    cache_folder: str = "cvm_dfp_cache",
    output_folder: str = "cvm_dfp_parquet",
) -> None:
    """Baixa e organiza dados anuais, gravando um arquivo Parquet por
    ano dentro de uma subpasta por tipo de documento. Anos já
    processados são detectados e pulados, permitindo retomar a
    execução depois de uma falha sem reprocessar tudo."""
    cache_path = Path(cache_folder)
    output_path = Path(output_folder)
    codigos_cvm = set(companies_cvm_codes)

    sessao = criar_sessao()

    for ano in range(first_year, last_year + 1):
        try:
            caminho_zip = baixar_zip_dfp(ano, cache_path, sessao)
        except requests.RequestException as erro:
            print(f"Ano {ano} indisponível no portal da CVM ({erro}), pulando.")
            continue

        for tipo in type_docs:
            for formato in type_format:
                pasta_saida = output_path / f"{tipo}_{formato}"
                pasta_saida.mkdir(parents=True, exist_ok=True)
                caminho_saida = pasta_saida / f"{ano}.parquet"

                if caminho_saida.exists():
                    continue  # ano já processado em execução anterior

                nome_arquivo = f"dfp_cia_aberta_{tipo}_{formato}_{ano}.csv"

                try:
                    df_ano = ler_csv_do_zip(caminho_zip, nome_arquivo, codigos_cvm)
                except Exception as erro:
                    print(f"Falha ao ler {nome_arquivo} ({erro}), pulando.")
                    continue

                if df_ano.empty:
                    continue

                tabela = pa.Table.from_pandas(df_ano, preserve_index=False)
                pq.write_table(tabela, caminho_saida)
                print(f"{ano} {tipo}_{formato}: {df_ano.shape[0]} linhas gravadas")

        caminho_zip.unlink(missing_ok=True)


if __name__ == "__main__":
    sessao_principal = criar_sessao()
    codigos_cvm = obter_codigos_ativos(sessao_principal)
    print(f"Total de companhias ativas: {len(codigos_cvm)}")

    get_dfp_data(
        companies_cvm_codes=codigos_cvm,
        first_year=2011,
        last_year=2025,
        type_docs=["DRE", "BPA", "BPP"],
        type_format=["con"],
    )

    # leitura sob demanda: pyarrow junta todos os arquivos da subpasta
    dre_con = pd.read_parquet("cvm_dfp_parquet/DRE_con")
    print(dre_con.shape)

In [ ]:
# 3.2 Dados trimestrais completos

import zipfile
from pathlib import Path
from typing import Iterable

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
import io
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_ITR_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/ITR/DADOS"
BASE_CAD_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/CAD/DADOS/cad_cia_aberta.csv"


def criar_sessao() -> requests.Session:
    
    sessao = requests.Session()
    retentativas = Retry(
        total=5,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
    )
    sessao.mount("https://", HTTPAdapter(max_retries=retentativas))
    return sessao


def obter_codigos_ativos(sessao: requests.Session) -> list[int]:
    resposta = sessao.get(BASE_CAD_URL, timeout=60)
    resposta.raise_for_status()
    cadastro = pd.read_csv(
        io.StringIO(resposta.content.decode("latin1")),
        sep=";",
    )
    ativas = cadastro.loc[cadastro["SIT"] == "ATIVO", "CD_CVM"]
    return ativas.unique().tolist()


def baixar_zip_itr(ano: int, cache_folder: Path, sessao: requests.Session) -> Path:
    
    cache_folder.mkdir(parents=True, exist_ok=True)
    destino = cache_folder / f"itr_cia_aberta_{ano}.zip"
    if destino.exists():
        return destino

    url = f"{BASE_ITR_URL}/itr_cia_aberta_{ano}.zip"
    with sessao.get(url, timeout=60, stream=True) as resposta:
        resposta.raise_for_status()
        with open(destino, "wb") as arquivo:
            for bloco in resposta.iter_content(chunk_size=1024 * 1024):
                arquivo.write(bloco)

    return destino


def ler_csv_do_zip(
    caminho_zip: Path,
    nome_arquivo: str,
    codigos_cvm: set[int],
) -> pd.DataFrame:
    with zipfile.ZipFile(caminho_zip) as arquivo_zip:
        if nome_arquivo not in arquivo_zip.namelist():
            return pd.DataFrame()

        with arquivo_zip.open(nome_arquivo) as csv_bytes:
            leitor = pd.read_csv(
                csv_bytes,
                sep=";",
                encoding="latin1",
                decimal=",",
                dtype={"CD_CVM": "Int64"},
                chunksize=200_000,
            )
            partes = [
                parte[parte["CD_CVM"].isin(codigos_cvm)] for parte in leitor
            ]

    if not partes:
        return pd.DataFrame()

    return pd.concat(partes, ignore_index=True)


def get_itr_data(
    companies_cvm_codes: Iterable[int],
    first_year: int = 2011,
    last_year: int = 2025,
    type_docs: list[str] = ["BPA", "BPP", "DRE"],
    type_format: list[str] = ["con", "ind"],
    cache_folder: str = "cvm_itr_cache",
    output_folder: str = "cvm_itr_parquet",
) -> None:
    
    cache_path = Path(cache_folder)
    output_path = Path(output_folder)
    codigos_cvm = set(companies_cvm_codes)

    sessao = criar_sessao()

    for ano in range(first_year, last_year + 1):
        try:
            caminho_zip = baixar_zip_itr(ano, cache_path, sessao)
        except requests.RequestException as erro:
            print(f"Ano {ano} indisponível no portal da CVM ({erro}), pulando.")
            continue

        for tipo in type_docs:
            for formato in type_format:
                pasta_saida = output_path / f"{tipo}_{formato}"
                pasta_saida.mkdir(parents=True, exist_ok=True)
                caminho_saida = pasta_saida / f"{ano}.parquet"

                if caminho_saida.exists():
                    continue  # ano já processado em execução anterior

                nome_arquivo = f"itr_cia_aberta_{tipo}_{formato}_{ano}.csv"

                try:
                    df_ano = ler_csv_do_zip(caminho_zip, nome_arquivo, codigos_cvm)
                except Exception as erro:
                    print(f"Falha ao ler {nome_arquivo} ({erro}), pulando.")
                    continue

                if df_ano.empty:
                    continue

                tabela = pa.Table.from_pandas(df_ano, preserve_index=False)
                pq.write_table(tabela, caminho_saida)
                print(f"{ano} {tipo}_{formato}: {df_ano.shape[0]} linhas gravadas")

        caminho_zip.unlink(missing_ok=True)


if __name__ == "__main__":
    sessao_principal = criar_sessao()
    codigos_cvm = obter_codigos_ativos(sessao_principal)
    print(f"Total de companhias ativas: {len(codigos_cvm)}")

    get_itr_data(
        companies_cvm_codes=codigos_cvm,
        first_year=2011,
        last_year=2025,
        type_docs=["DRE", "BPA", "BPP"],
        type_format=["con"],
    )

    # leitura sob demanda: pyarrow junta todos os arquivos da subpasta
    dre_con = pd.read_parquet("cvm_itr_parquet/DRE_con")
    print(dre_con.shape)

In [ ]:
# 3.3 Exportação (anual e trimestral)

from pathlib import Path

import pandas as pd


def exportar_para_csv(pasta_parquet: str, pasta_saida: str) -> None:
    
    pasta = Path(pasta_parquet)
    subpastas = sorted(p for p in pasta.iterdir() if p.is_dir())

    if not subpastas:
        print(f"Nenhuma subpasta encontrada em {pasta_parquet}.")
        return

    Path(pasta_saida).mkdir(parents=True, exist_ok=True)

    for subpasta in subpastas:
        arquivos_parquet = sorted(subpasta.glob("*.parquet"))
        if not arquivos_parquet:
            print(f"{subpasta.name}: nenhum arquivo Parquet encontrado, pulando.")
            continue

        df = pd.read_parquet(subpasta)
        caminho_csv = Path(pasta_saida) / f"{subpasta.name}.csv"
        df.to_csv(caminho_csv, index=False)
        print(f"{subpasta.name}: {df.shape[0]} linhas exportadas para {caminho_csv}")


if __name__ == "__main__":
    exportar_para_csv("cvm_itr_parquet", "dados_trim_completo")
    exportar_para_csv("cvm_dfp_parquet", "dados_anuais_completo")

# Obs.: os arquivos Parquet de origem vêm das pastas cvm_itr_parquet
# e cvm_dfp_parquet, geradas pelas células 3.1 e 3.2. Cada subpasta
# (ex.: DRE_con, BPA_con) reúne um arquivo por ano, e vira um único
# CSV consolidado na pasta de saída.

### **FIM**